# AI应用基础（48课时版） · 工单M3-1《ResNet迁移学习》
**第7周 | 2课时(80分钟) | 理实一体**  | 学生实训版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 创新优化；示范讲解 |
| **B 组长** | 标准达成 | F1≥0.88 |
| **C 操作员** | 基础完成 | F1≥0.72（2h内） |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）


## 一、课前资讯（Predict·预测）
> **预习快问**：['W7课前测①：本课工单M2-2的及格线是什么？（答：F1≥0.72（2h内））', "W7课前测②：'集成思想'用一句话说说你的理解"]

> **课前检查**：导学单+预习自测已完成；data/数据已就位、starter可运行（--check环境OK）

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


## 三、任务要求（Make）
**实施步骤**
1. 回顾预习卡与微课
2. 知识精讲（先例后理）
3. 演示代码骨架
4. 基础实现（D/C层保底）
5. 小结+布置第2次课

**验收标准**：及格(D)=`F1≥0.72（2h内）` ；优秀(B+)=`F1≥0.88`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
# -*- coding: utf-8 -*-
"""M3-1 迁移学习 starter（教学周7）：冻结预训练特征提取器+只训分类头——零torch理解ResNet精髓

思想实验（离线CPU版）：
  "预训练" = 用2000条无标注数据学出紧凑特征(PCA)——学完冻结，不随任务更新
  迁移 = 冻结它，只用16条标注样本训顶端逻辑回归头（小样本逆袭）
  对照 = 不用预训练特征，16条样本直接在192维(多为噪声维)训头——维度灾难大概率崩
真机版( torchvision ResNet18冻结实验 )见教师GPU演示——概念在本文件。
"""
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

### Starter 2：`def make_world(seed=7, n_dim=192):`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def make_world(seed=7, n_dim=192):
    """合成工业特征：8种工况原型仅前8个测点携带信息，其余为环境噪声维"""
    rng = np.random.RandomState(seed)
    centers = rng.normal(0, 3, (8, n_dim))
    centers[:, 8:] = 0
    big = np.vstack([c + rng.normal(0, 1, (250, n_dim)) for c in centers])   # 海量"无标注"
    small = np.vstack([c + rng.normal(0, 1, (2, n_dim)) for c in centers])   # 仅16条有标注
    test = np.vstack([c + rng.normal(0, 1, (25, n_dim)) for c in centers])   # 200条测试
    label = lambda X: (X[:, 0] > 0).astype(int)      # 真实规律藏在第0维
    return big, small, test, label(small), label(test)

### Starter 3：`class FrozenExtractor:`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
class FrozenExtractor:
    """'预训练特征提取器'：在海量数据上一次学成(PCA)，之后冻结不再更新"""

    def __init__(self, big, dim=8):
        self.pca = PCA(n_components=dim, random_state=0).fit(big)  # 预训练=在海量数据上学结构

    def transform(self, X):
        # TODO-W7-1: 返回冻结特征 self.pca.transform(X)（只前传、不更新权重）
        return X

### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - TODO-W7-1: 返回冻结特征
>   self.pca.transform(X)（只前传、不更新权重）
> - TODO-W7-2: Xf = ext.transform(small)
>   训头，测试集评估，得 s_transfer

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位TODO行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改参数/换实现对比效果，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方"验收标准"，及格(D)=`F1≥0.72（2h内）`，优秀(B+)=`F1≥0.88`

### Starter 4：`def main():`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def main():
    big, small, test, y_small, y_test = make_world()
    ext = FrozenExtractor(big)
    # 对照组：16条样本直接在192维原始空间训头
    s_raw = LogisticRegression(max_iter=500).fit(small, y_small).score(test, y_test)
    # 迁移组：冻结特征 → 训头
    # TODO-W7-2: Xf = ext.transform(small) 训头，测试集评估，得 s_transfer
    s_transfer = None
    print(f"对照(16条直连192维训头): 测试={s_raw:.2f}")
    if s_transfer is not None:
        print(f"迁移(冻结特征+训头):     测试={s_transfer:.2f}")
        print("目标：迁移组≥对照组 —— 预训练特征是维度灾难的解药（写进冻结/微调取舍实验）")
    else:
        print("完成TODO-W7-1/2看对比——两个TODO各一行")

### Starter 5：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
if __name__ == "__main__":
    main()

## 四、提交要求与避坑
- 提交: `学号_姓名_M3-1.py` 或本notebook(含输出)
